# Cohere reranking (rerank-v4.0-pro)

This notebook reads the latest retrieval evaluation output (`rankings.jsonl`) and reranks **all candidates** per query using Cohere.

Prereqs:
- Set `COHERE_API_KEY` in your environment.
- Ensure you downloaded the Processing eval output so `test_results/runs/rankings.jsonl` exists.


In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

modernbert_dir = Path("../modernbert").resolve()
sys.path.insert(0, str(modernbert_dir))

print("modernbert_dir:", modernbert_dir)
print("COHERE_API_KEY set:", bool(os.environ.get("COHERE_API_KEY")))

modernbert_dir: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/modernbert
COHERE_API_KEY set: True


In [2]:
from reranking.cohere_reranker import CohereReranker
from reranking.rerank_rankings import print_topk_for_query, rerank_rankings_file

# Paths
# - this assumes you downloaded the evaluation output into `corporate_reorganization/test_results/`.
rankings_jsonl = Path("../test_results/runs/rankings.jsonl").resolve()
processed_dir = Path("../data/final_annotations_gold/processed").resolve()
reranked_out = Path("../test_results/runs/cohere_reranked.jsonl").resolve()

assert rankings_jsonl.exists(), f"Missing rankings file: {rankings_jsonl}"
assert processed_dir.exists(), f"Missing processed_dir: {processed_dir}"

print("rankings_jsonl:", rankings_jsonl)
print("processed_dir:", processed_dir)
print("reranked_out:", reranked_out)


rankings_jsonl: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/test_results/runs/rankings.jsonl
processed_dir: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/data/final_annotations_gold/processed
reranked_out: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/test_results/runs/cohere_reranked.jsonl


In [3]:
# Cohere config
# Note: if Cohere enforces a per-request document limit, the reranker will chunk and merge by score.
reranker = CohereReranker.from_env(
    model="rerank-v4.0-pro",
    max_documents_per_request=100,
)

rerank_rankings_file(
    processed_dir=processed_dir,
    rankings_jsonl=rankings_jsonl,
    split="test",
    input_system="fine_tuned",
    output_path=reranked_out,
    reranker=reranker,
    max_docs_per_request=100,
)

print("Wrote:", reranked_out)


/home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Wrote: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/test_results/runs/cohere_reranked.jsonl


In [1]:
# Print top-1/5/10/20 for the first query in the reranked file
import json

with reranked_out.open("r", encoding="utf-8") as f:
    first_row = json.loads(next(f))

print_topk_for_query(
    reranked_jsonl=reranked_out,
    query_id=first_row["query_id"],
    ks=(1, 5, 10, 20),
)


NameError: name 'reranked_out' is not defined

In [2]:
from reranking.eval_metrics import evaluate_reranked_jsonl, format_summary_table, candidate_stats_from_reranked_jsonl, mrr_full_from_reranked_jsonl

summary = evaluate_reranked_jsonl(
      reranked_jsonl=Path("../test_results/runs/cohere_reranked.jsonl"),
      ks=(1, 5, 10, 20),
  )

stats = candidate_stats_from_reranked_jsonl(reranked_jsonl="../test_results/runs/cohere_reranked.jsonl")
print("Candidate stats:")
print("num_queries:", stats["num_queries"])
print("avg_candidate pool size:", stats["avg_candidates"])

out = mrr_full_from_reranked_jsonl(reranked_jsonl="../test_results/runs/cohere_reranked.jsonl")

print("MRR:", out)

print(format_summary_table(summary, ks=(1,5, 10, 20)))

Candidate stats:
num_queries: 40
avg_candidate pool size: 138.8
MRR: {'mrr_full': 0.15817767183369918, 'num_queries': 40.0}
GLOBAL
- recall_at_1: 0.0750
- mrr_at_1: 0.0750
- set_recall_at_1: 0.0500
- exact_set_match_at_1: 0.0250
- recall_at_5: 0.2750
- mrr_at_5: 0.1233
- set_recall_at_5: 0.1237
- exact_set_match_at_5: 0.0250
- recall_at_10: 0.3750
- mrr_at_10: 0.1361
- set_recall_at_10: 0.1883
- exact_set_match_at_10: 0.0500
- recall_at_20: 0.5250
- mrr_at_20: 0.1464
- set_recall_at_20: 0.3192
- exact_set_match_at_20: 0.1500

BY DOC_ID
  doc_id=37  n=9  R@1=0.1111  MRR@1=0.1111  R@5=0.3333  MRR@5=0.1611  R@10=0.4444  MRR@10=0.1722  R@20=0.7778  MRR@20=0.1978
  doc_id=46  n=13  R@1=0.1538  MRR@1=0.1538  R@5=0.3077  MRR@5=0.1846  R@10=0.4615  MRR@10=0.2033  R@20=0.5385  MRR@20=0.2078
  doc_id=65  n=6  R@1=0.0000  MRR@1=0.0000  R@5=0.5000  MRR@5=0.1389  R@10=0.6667  MRR@10=0.1667  R@20=0.6667  MRR@20=0.1667
  doc_id=96  n=12  R@1=0.0000  MRR@1=0.0000  R@5=0.0833  MRR@5=0.0208  R@10=0.0833